trains Linear Regression, Random Forest, and XGBoost on:

1. Non-PCA features
2. PCA features

In [13]:
%pip install xgboost

Note: you may need to restart the kernel to use updated packages.


## 1. Imports and paths

In [18]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use("Agg")  # safe for notebook saving

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.dummy import DummyRegressor

try:
    from xgboost import XGBRegressor
    XGBOOST_AVAILABLE = True
    print("XGBoost imported successfully.")
except Exception as e:
    XGBOOST_AVAILABLE = False
    print("XGBoost import failed:", e)

PROJECT_DIR = Path("/Users/arjun/Documents/UAV thesis/uav-thesis/uav_px4_dataset_project")

NO_DR_DIR  = PROJECT_DIR / "model_ready_no_dr"
PCA_DIR    = PROJECT_DIR / "model_ready_with_pca"
OUTPUT_DIR = PROJECT_DIR / "model_outputs" / "baseline_models"
PLOT_DIR   = OUTPUT_DIR / "plots"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

target_col = "target_energy_next_60s_wh"
id_cols    = ["log_id", "dataset_label"]

print("Output directory:", OUTPUT_DIR)
print("Plot directory  :", PLOT_DIR)

XGBoost imported successfully.
Output directory: /Users/arjun/Documents/UAV thesis/uav-thesis/uav_px4_dataset_project/model_outputs/baseline_models
Plot directory  : /Users/arjun/Documents/UAV thesis/uav-thesis/uav_px4_dataset_project/model_outputs/baseline_models/plots


## 2. Helper functions

In [19]:
# ── metric helpers ────────────────────────────────────────────────────────────

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))


def get_feature_cols(df, target_col, id_cols):
    return [c for c in df.columns if c not in id_cols + [target_col]]


# ── core evaluation ───────────────────────────────────────────────────────────

def evaluate_model(model_name, dataset_name, model,
                   X_train, y_train,
                   X_val,   y_val,
                   X_test,  y_test,
                   test_df=None):
    """
    Fits model, returns a results dict with MAE / RMSE / R² for val + test,
    plus per-label test metrics and flight-level test MAE if test_df supplied.
    """
    print(f"\n{'='*60}")
    print(f"  Training : {model_name}  |  Dataset : {dataset_name}")
    print(f"{'='*60}")

    model.fit(X_train, y_train)

    val_pred  = model.predict(X_val)
    test_pred = model.predict(X_test)

    results = {
        "dataset"   : dataset_name,
        "model"     : model_name,
        # --- validation ---
        "val_MAE"   : mean_absolute_error(y_val, val_pred),
        "val_RMSE"  : rmse(y_val, val_pred),
        "val_R2"    : r2_score(y_val, val_pred),
        # --- test overall ---
        "test_MAE"  : mean_absolute_error(y_test, test_pred),
        "test_RMSE" : rmse(y_test, test_pred),
        "test_R2"   : r2_score(y_test, test_pred),
    }

    # ── per-label test breakdown ──────────────────────────────────────────────
    if test_df is not None:
        for label in ["normal", "abnormal"]:
            mask = test_df["dataset_label"].values == label
            if mask.sum() == 0:
                continue
            yt = y_test.values[mask]
            yp = test_pred[mask]
            results[f"test_MAE_{label}"]  = mean_absolute_error(yt, yp)
            results[f"test_RMSE_{label}"] = rmse(yt, yp)
            results[f"test_R2_{label}"]   = r2_score(yt, yp)
            print(f"  [{label:8s}]  MAE={results[f'test_MAE_{label}']:.4f}  "
                  f"RMSE={results[f'test_RMSE_{label}']:.4f}  "
                  f"R²={results[f'test_R2_{label}']:.4f}")

    # ── flight-level MAE ──────────────────────────────────────────────────────
    if test_df is not None and "log_id" in test_df.columns:
        tmp = test_df[["log_id"]].copy()
        tmp["y_true"] = y_test.values
        tmp["y_pred"] = test_pred
        flight_mae = (
            tmp.groupby("log_id")
               .apply(lambda g: mean_absolute_error(g["y_true"], g["y_pred"]))
               .reset_index()
               .rename(columns={0: "flight_MAE"})
        )
        results["test_flight_MAE_mean"]   = flight_mae["flight_MAE"].mean()
        results["test_flight_MAE_median"] = flight_mae["flight_MAE"].median()
        print(f"  [flight]   mean MAE={results['test_flight_MAE_mean']:.4f}  "
              f"median MAE={results['test_flight_MAE_median']:.4f}")

    print(f"  [overall]  "
          f"test MAE={results['test_MAE']:.4f}  "
          f"RMSE={results['test_RMSE']:.4f}  "
          f"R²={results['test_R2']:.4f}")

    return results, model, test_pred


# ── diagnostic plots ──────────────────────────────────────────────────────────

def save_diagnostic_plots(model_name, dataset_name, y_test, test_pred, plot_dir):
    """
    Saves predicted-vs-actual scatter and residual distribution.
    """
    safe = f"{dataset_name}__{model_name}".lower().replace(" ", "_").replace("/", "_")
    residuals = y_test.values - test_pred

    # 1. Predicted vs Actual
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.scatter(y_test, test_pred, alpha=0.15, s=4, color="steelblue")
    lim_max = max(y_test.max(), test_pred.max()) * 1.05
    ax.plot([0, lim_max], [0, lim_max], "r--", lw=1.5, label="Perfect prediction")
    ax.set_xlabel("Actual Energy (Wh)")
    ax.set_ylabel("Predicted Energy (Wh)")
    ax.set_title(f"Predicted vs Actual\n{model_name} | {dataset_name}")
    ax.legend()
    plt.tight_layout()
    fig.savefig(plot_dir / f"pred_vs_actual__{safe}.png", dpi=150)
    plt.close(fig)

    # 2. Residual distribution
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.hist(residuals, bins=80, color="coral", edgecolor="none", alpha=0.8)
    ax.axvline(0, color="black", lw=1.2, linestyle="--")
    ax.set_xlabel("Residual (Actual − Predicted) Wh")
    ax.set_ylabel("Count")
    ax.set_title(f"Residual Distribution\n{model_name} | {dataset_name}")
    plt.tight_layout()
    fig.savefig(plot_dir / f"residuals__{safe}.png", dpi=150)
    plt.close(fig)

    # 3. Residuals vs Predicted (heteroscedasticity check)
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.scatter(test_pred, residuals, alpha=0.10, s=4, color="slategray")
    ax.axhline(0, color="red", lw=1.2, linestyle="--")
    ax.set_xlabel("Predicted Energy (Wh)")
    ax.set_ylabel("Residual (Wh)")
    ax.set_title(f"Residuals vs Predicted\n{model_name} | {dataset_name}")
    plt.tight_layout()
    fig.savefig(plot_dir / f"resid_vs_pred__{safe}.png", dpi=150)
    plt.close(fig)

    print(f"  Plots saved → {safe}")


# ── feature importance ────────────────────────────────────────────────────────

def save_feature_importance(model, model_name, dataset_name, feature_cols, plot_dir, top_n=20):
    """
    Works for RandomForest and XGBoost. Saves ranked bar chart + CSV.
    """
    if not hasattr(model, "feature_importances_"):
        return

    safe = f"{dataset_name}__{model_name}".lower().replace(" ", "_").replace("/", "_")
    imp  = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)
    imp_top = imp.head(top_n)

    fig, ax = plt.subplots(figsize=(8, max(4, top_n * 0.35)))
    imp_top[::-1].plot(kind="barh", ax=ax, color="teal")
    ax.set_xlabel("Feature Importance (MDI)")
    ax.set_title(f"Top {top_n} Feature Importances\n{model_name} | {dataset_name}")
    plt.tight_layout()
    fig.savefig(plot_dir / f"feat_imp__{safe}.png", dpi=150)
    plt.close(fig)

    imp.to_csv(plot_dir / f"feat_imp__{safe}.csv", header=["importance"])
    print(f"  Feature importance saved → feat_imp__{safe}")

## 3. Load non-PCA data

In [20]:
train_no_dr = pd.read_csv(NO_DR_DIR / "train_no_dr_imputed.csv")
val_no_dr   = pd.read_csv(NO_DR_DIR / "val_no_dr_imputed.csv")
test_no_dr  = pd.read_csv(NO_DR_DIR / "test_no_dr_imputed.csv")

feature_cols_no_dr = get_feature_cols(train_no_dr, target_col, id_cols)

X_train_no_dr = train_no_dr[feature_cols_no_dr]
X_val_no_dr   = val_no_dr[feature_cols_no_dr]
X_test_no_dr  = test_no_dr[feature_cols_no_dr]

y_train_no_dr = train_no_dr[target_col]
y_val_no_dr   = val_no_dr[target_col]
y_test_no_dr  = test_no_dr[target_col]

print("Non-PCA train shape:", X_train_no_dr.shape)
print("Non-PCA features   :", feature_cols_no_dr)

Non-PCA train shape: (694549, 11)
Non-PCA features   : ['battery_status__cell_count', 'battery_status__current_a', 'battery_status__current_filtered_a', 'battery_status__discharged_mah', 'battery_status__remaining', 'battery_status__scale', 'battery_status__voltage_filtered_v', 'battery_status__voltage_v', 'cumulative_energy_wh', 'power_w', 'time_s']


## 4. Load PCA data

In [21]:
train_pca = pd.read_csv(PCA_DIR / "train_pca_95.csv")
val_pca   = pd.read_csv(PCA_DIR / "val_pca_95.csv")
test_pca  = pd.read_csv(PCA_DIR / "test_pca_95.csv")

feature_cols_pca = get_feature_cols(train_pca, target_col, id_cols)

X_train_pca = train_pca[feature_cols_pca]
X_val_pca   = val_pca[feature_cols_pca]
X_test_pca  = test_pca[feature_cols_pca]

y_train_pca = train_pca[target_col]
y_val_pca   = val_pca[target_col]
y_test_pca  = test_pca[target_col]

print("PCA train shape  :", X_train_pca.shape)
print("PCA feature count:", len(feature_cols_pca))
print("PCA features[:10]:", feature_cols_pca[:10])

PCA train shape  : (694549, 6)
PCA feature count: 6
PCA features[:10]: ['PC1', 'PC2', 'PC3', 'PC4', 'PC5', 'PC6']


## 5. Naive baseline
Mean predictor naive baseline
Predicts the training-set mean for every sample.
This is the floor every model must beat to be considered useful.

In [22]:
naive_results = []

for dataset_name, (y_tr, y_va, y_te, test_df) in {
    "non_pca_features": (y_train_no_dr, y_val_no_dr, y_test_no_dr, test_no_dr),
    "pca_features"    : (y_train_pca,   y_val_pca,   y_test_pca,   test_pca),
}.items():

    dummy = DummyRegressor(strategy="mean")
    dummy.fit(np.zeros((len(y_tr), 1)), y_tr)   # DummyRegressor ignores X

    val_pred_naive  = np.full(len(y_va), y_tr.mean())
    test_pred_naive = np.full(len(y_te), y_tr.mean())

    row = {
        "dataset"   : dataset_name,
        "model"     : "Mean Predictor (Naive Baseline)",
        "val_MAE"   : mean_absolute_error(y_va, val_pred_naive),
        "val_RMSE"  : rmse(y_va, val_pred_naive),
        "val_R2"    : r2_score(y_va, val_pred_naive),
        "test_MAE"  : mean_absolute_error(y_te, test_pred_naive),
        "test_RMSE" : rmse(y_te, test_pred_naive),
        "test_R2"   : r2_score(y_te, test_pred_naive),
    }

    # per-label breakdown
    for label in ["normal", "abnormal"]:
        mask = test_df["dataset_label"].values == label
        if mask.sum() == 0:
            continue
        yt = y_te.values[mask]
        yp = test_pred_naive[mask]
        row[f"test_MAE_{label}"]  = mean_absolute_error(yt, yp)
        row[f"test_RMSE_{label}"] = rmse(yt, yp)
        row[f"test_R2_{label}"]   = r2_score(yt, yp)

    # flight-level MAE
    tmp = test_df[["log_id"]].copy()
    tmp["y_true"] = y_te.values
    tmp["y_pred"] = test_pred_naive
    fl_mae = (
        tmp.groupby("log_id")
           .apply(lambda g: mean_absolute_error(g["y_true"], g["y_pred"]))
    )
    row["test_flight_MAE_mean"]   = fl_mae.mean()
    row["test_flight_MAE_median"] = fl_mae.median()

    naive_results.append(row)
    print(f"Naive baseline [{dataset_name}]  "
          f"test MAE={row['test_MAE']:.4f}  R²={row['test_R2']:.4f}")

print("\nNaive baseline complete. Every trained model must beat these numbers.")

Naive baseline [non_pca_features]  test MAE=1.5109  R²=-0.1036
Naive baseline [pca_features]  test MAE=1.5109  R²=-0.1036

Naive baseline complete. Every trained model must beat these numbers.


/var/folders/dg/72q0h6ss3xq0zf641p8b9gnm0000gn/T/ipykernel_7081/3221600561.py:42: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: mean_absolute_error(g["y_true"], g["y_pred"]))
/var/folders/dg/72q0h6ss3xq0zf641p8b9gnm0000gn/T/ipykernel_7081/3221600561.py:42: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: mean_absolute_error(g["y_true"], g["y_pred"]))


## 6.  Train all models with full diagnostics

In [24]:
all_results    = list(naive_results)   # start with naive baseline already included
trained_models = {}

datasets = {
    "non_pca_features": (
        X_train_no_dr, y_train_no_dr,
        X_val_no_dr,   y_val_no_dr,
        X_test_no_dr,  y_test_no_dr,
        test_no_dr,           # full test df for label/flight breakdown
        feature_cols_no_dr,   # for feature importance
    ),
    "pca_features": (
        X_train_pca, y_train_pca,
        X_val_pca,   y_val_pca,
        X_test_pca,  y_test_pca,
        test_pca,
        feature_cols_pca,
    ),
}

for dataset_name, (X_tr, y_tr, X_va, y_va, X_te, y_te, test_df, feat_cols) in datasets.items():

    models = {
        "Linear Regression": LinearRegression(),
        "Random Forest": RandomForestRegressor(
            n_estimators=200,
            max_depth=None,
            random_state=42,
            n_jobs=-1
        ),
    }

    if XGBOOST_AVAILABLE:
        models["XGBoost"] = XGBRegressor(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=6,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="reg:squarederror",
            random_state=42,
            n_jobs=-1,
            verbosity=0,
        )

    for model_name, model in models.items():

        # ── train + evaluate ──────────────────────────────────────────────────
        results, fitted_model, test_pred = evaluate_model(
            model_name, dataset_name, model,
            X_tr, y_tr,
            X_va, y_va,
            X_te, y_te,
            test_df=test_df,
        )
        all_results.append(results)
        trained_models[f"{dataset_name}__{model_name}"] = fitted_model

        # ── diagnostic plots ──────────────────────────────────────────────────
        save_diagnostic_plots(model_name, dataset_name, y_te, test_pred, PLOT_DIR)

        # ── feature importance (RF and XGBoost only) ──────────────────────────
        save_feature_importance(fitted_model, model_name, dataset_name,
                                feat_cols, PLOT_DIR, top_n=20)

print("\n\nAll models trained.")


  Training : Linear Regression  |  Dataset : non_pca_features
  [normal  ]  MAE=0.3408  RMSE=0.8883  R²=0.8164
  [abnormal]  MAE=0.2681  RMSE=0.4637  R²=0.8541
  [flight]   mean MAE=0.2825  median MAE=0.2149
  [overall]  test MAE=0.3208  RMSE=0.7944  R²=0.8212


/var/folders/dg/72q0h6ss3xq0zf641p8b9gnm0000gn/T/ipykernel_7081/3299518384.py:66: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: mean_absolute_error(g["y_true"], g["y_pred"]))


  Plots saved → non_pca_features__linear_regression

  Training : Random Forest  |  Dataset : non_pca_features
  [normal  ]  MAE=0.4115  RMSE=0.9931  R²=0.7705
  [abnormal]  MAE=0.1481  RMSE=0.3454  R²=0.9190
  [flight]   mean MAE=0.2749  median MAE=0.1756
  [overall]  test MAE=0.3390  RMSE=0.8647  R²=0.7882


/var/folders/dg/72q0h6ss3xq0zf641p8b9gnm0000gn/T/ipykernel_7081/3299518384.py:66: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: mean_absolute_error(g["y_true"], g["y_pred"]))


  Plots saved → non_pca_features__random_forest
  Feature importance saved → feat_imp__non_pca_features__random_forest

  Training : XGBoost  |  Dataset : non_pca_features
  [normal  ]  MAE=0.3804  RMSE=0.9804  R²=0.7764
  [abnormal]  MAE=0.1770  RMSE=0.3288  R²=0.9266
  [flight]   mean MAE=0.2507  median MAE=0.1550
  [overall]  test MAE=0.3244  RMSE=0.8523  R²=0.7942


...

  Plots saved → pca_features__linear_regression

  Training : Random Forest  |  Dataset : pca_features
  [normal  ]  MAE=0.4922  RMSE=1.0757  R²=0.7308
  [abnormal]  MAE=0.2093  RMSE=0.4227  R²=0.8787
  [flight]   mean MAE=0.3542  median MAE=0.2607
  [overall]  test MAE=0.4144  RMSE=0.9423  R²=0.7485


/var/folders/dg/72q0h6ss3xq0zf641p8b9gnm0000gn/T/ipykernel_7081/3299518384.py:66: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: mean_absolute_error(g["y_true"], g["y_pred"]))


  Plots saved → pca_features__random_forest
  Feature importance saved → feat_imp__pca_features__random_forest

  Training : XGBoost  |  Dataset : pca_features
  [normal  ]  MAE=0.4645  RMSE=1.0738  R²=0.7318
  [abnormal]  MAE=0.2213  RMSE=0.3941  R²=0.8946
  [flight]   mean MAE=0.3248  median MAE=0.1985
  [overall]  test MAE=0.3976  RMSE=0.9373  R²=0.7512


/var/folders/dg/72q0h6ss3xq0zf641p8b9gnm0000gn/T/ipykernel_7081/3299518384.py:66: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: mean_absolute_error(g["y_true"], g["y_pred"]))


  Plots saved → pca_features__xgboost
  Feature importance saved → feat_imp__pca_features__xgboost


All models trained.


## 8. Results table

In [25]:
baseline_results_df = pd.DataFrame(all_results)

# ── display full table ────────────────────────────────────────────────────────
display(baseline_results_df)

# ── display clean summary (thesis-ready columns only) ────────────────────────
summary_cols = [
    "dataset", "model",
    "test_MAE", "test_RMSE", "test_R2",
    "test_MAE_normal", "test_MAE_abnormal",
    "test_flight_MAE_mean",
]
summary_cols_present = [c for c in summary_cols if c in baseline_results_df.columns]
print("\n── Thesis Summary Table ──────────────────────────────────────────")
display(baseline_results_df[summary_cols_present].round(4))

# ── save ──────────────────────────────────────────────────────────────────────
baseline_results_df.to_csv(OUTPUT_DIR / "baseline_model_results_full.csv", index=False)
baseline_results_df[summary_cols_present].to_csv(
    OUTPUT_DIR / "baseline_model_results_summary.csv", index=False
)
print("\nResults saved.")

,dataset,model,val_MAE,val_RMSE,val_R2,test_MAE,test_RMSE,test_R2,test_MAE_normal,test_RMSE_normal,test_R2_normal,test_MAE_abnormal,test_RMSE_abnormal,test_R2_abnormal,test_flight_MAE_mean,test_flight_MAE_median
0,non_pca_features,Mean Predictor (Naive Baseline),1.741930,2.066421,-0.062281,1.510946,1.973825,-0.103618,1.628336,2.143791,-0.069221,1.201731,1.432662,-0.393212,1.410577,1.380149
1,pca_features,Mean Predictor (Naive Baseline),1.741930,2.066421,-0.062281,1.510946,1.973825,-0.103618,1.628336,2.143791,-0.069221,1.201731,1.432662,-0.393212,1.410577,1.380149
2,non_pca_features,Linear Regression,0.335030,0.663986,0.890322,0.320828,0.794404,0.821234,0.340839,0.888275,0.816432,0.268119,0.463694,0.854054,0.282534,0.214933
3,non_pca_features,Random Forest,0.513812,0.984216,0.759019,0.339027,0.864700,0.788197,0.411509,0.993114,0.770543,0.148103,0.345394,0.919023,0.274933,0.175604
4,non_pca_features,XGBoost,0.484644,1.375005,0.529662,0.324419,0.852343,0.794207,0.380387,0.980429,0.776367,0.176997,0.328838,0.926601,0.250717,0.155033
5,pca_features,Linear Regression,0.602797,0.907694,0.795035,0.609220,1.074016,0.673245,0.661447,1.195175,0.667673,0.471648,0.655222,0.708588,0.535075,0.426202
6,pca_features,Random Forest,0.505894,0.955011,0.773109,0.414374,0.942300,0.748476,0.492243,1.075730,0.730779,0.209259,0.422680,0.878730,0.354162,0.260669
7,pca_features,XGBoost,0.517864,1.016906,0.742745,0.397594,0.937266,0.751156,0.464534,1.073783,0.731752,0.221270,0.394077,0.894587,0.324831,0.198539



── Thesis Summary Table ──────────────────────────────────────────


,dataset,model,test_MAE,test_RMSE,test_R2,test_MAE_normal,test_MAE_abnormal,test_flight_MAE_mean
0,non_pca_features,Mean Predictor (Naive Baseline),1.5109,1.9738,-0.1036,1.6283,1.2017,1.4106
1,pca_features,Mean Predictor (Naive Baseline),1.5109,1.9738,-0.1036,1.6283,1.2017,1.4106
2,non_pca_features,Linear Regression,0.3208,0.7944,0.8212,0.3408,0.2681,0.2825
3,non_pca_features,Random Forest,0.3390,0.8647,0.7882,0.4115,0.1481,0.2749
4,non_pca_features,XGBoost,0.3244,0.8523,0.7942,0.3804,0.1770,0.2507
5,pca_features,Linear Regression,0.6092,1.0740,0.6732,0.6614,0.4716,0.5351
6,pca_features,Random Forest,0.4144,0.9423,0.7485,0.4922,0.2093,0.3542
7,pca_features,XGBoost,0.3976,0.9373,0.7512,0.4645,0.2213,0.3248



Results saved.


## 9. Save models

In [27]:
for name, model in trained_models.items():
    safe_name = name.lower().replace(" ", "_").replace("/", "_")
    joblib.dump(model, OUTPUT_DIR / f"{safe_name}.joblib")
    print(f"Saved: {safe_name}.joblib")

print("\nAll models saved.")

Saved: non_pca_features__linear_regression.joblib
Saved: non_pca_features__random_forest.joblib
Saved: non_pca_features__xgboost.joblib
Saved: pca_features__linear_regression.joblib
Saved: pca_features__random_forest.joblib
Saved: pca_features__xgboost.joblib

All models saved.


## 10. Summary 

In [ ]:
print("\n" + "="*70)
print("BASELINE RESULTS SUMMARY")
print("="*70)

for _, row in baseline_results_df[summary_cols_present].iterrows():
    print(f"\n  {row['model']} | {row['dataset']}")
    print(f"    Overall  →  MAE: {row['test_MAE']:.4f}  "
          f"RMSE: {row['test_RMSE']:.4f}  R²: {row['test_R2']:.4f}")
    if "test_MAE_normal" in row:
        print(f"    Normal   →  MAE: {row['test_MAE_normal']:.4f}")
    if "test_MAE_abnormal" in row:
        print(f"    Abnormal →  MAE: {row['test_MAE_abnormal']:.4f}")
    if "test_flight_MAE_mean" in row:
        print(f"    Flight   →  mean MAE: {row['test_flight_MAE_mean']:.4f}")

print("\n" + "="*70)
print(f"Plots saved to : {PLOT_DIR}")
print(f"Results saved to: {OUTPUT_DIR}")
print("="*70)


BASELINE RESULTS SUMMARY — for thesis notes

  Mean Predictor (Naive Baseline) | non_pca_features
    Overall  →  MAE: 1.5109  RMSE: 1.9738  R²: -0.1036
    Normal   →  MAE: 1.6283
    Abnormal →  MAE: 1.2017
    Flight   →  mean MAE: 1.4106

  Mean Predictor (Naive Baseline) | pca_features
    Overall  →  MAE: 1.5109  RMSE: 1.9738  R²: -0.1036
    Normal   →  MAE: 1.6283
    Abnormal →  MAE: 1.2017
    Flight   →  mean MAE: 1.4106

  Linear Regression | non_pca_features
    Overall  →  MAE: 0.3208  RMSE: 0.7944  R²: 0.8212
    Normal   →  MAE: 0.3408
    Abnormal →  MAE: 0.2681
    Flight   →  mean MAE: 0.2825

  Random Forest | non_pca_features
    Overall  →  MAE: 0.3390  RMSE: 0.8647  R²: 0.7882
    Normal   →  MAE: 0.4115
    Abnormal →  MAE: 0.1481
    Flight   →  mean MAE: 0.2749

  XGBoost | non_pca_features
    Overall  →  MAE: 0.3244  RMSE: 0.8523  R²: 0.7942
    Normal   →  MAE: 0.3804
    Abnormal →  MAE: 0.1770
    Flight   →  mean MAE: 0.2507

  Linear Regression | pca_f